In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import time
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

In [ ]:
df_weather = pd.read_csv('../data_file/hanoi_weather_history.csv')
df_weather.describe()


,temp,app_temp,rh,wind_spd,wind_dir,pres,vis,clouds,precip,uv,dewpt
count,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24017.000000,24024.000000,24024.000000,24024.000000,24024.000000
mean,25.323810,28.380303,78.461455,1.679912,175.933899,1009.663836,10.252321,69.429820,0.318248,1.321932,20.970051
std,5.455107,8.461583,14.985092,0.847098,97.484502,7.398373,2.674902,32.881189,1.143260,1.918951,5.954365
min,7.000000,6.600000,17.000000,0.000000,0.000000,983.000000,0.000000,0.000000,0.000000,0.000000,-6.900000
25%,21.500000,21.800000,69.000000,1.000000,116.000000,1004.000000,10.000000,50.000000,0.000000,0.000000,17.700000
50%,26.200000,27.100000,83.000000,1.660000,142.000000,1009.000000,10.000000,79.000000,0.000000,0.600000,23.300000
75%,29.200000,35.200000,91.000000,2.000000,245.000000,1015.000000,10.000000,100.000000,0.000000,2.100000,25.400000
max,40.000000,52.700000,100.000000,12.400000,360.000000,1033.000000,16.000000,100.000000,40.000000,12.100000,29.600000


| Cột          | Mô tả                                                                                       |
|--------------|---------------------------------------------------------------------------------------------|
| `datetime`   | Thời gian ghi dữ liệu, dạng chuỗi (object), chứa thông tin về ngày và giờ.                 |
| `temp`       | Nhiệt độ thực tế (độ C), kiểu dữ liệu `float64`.                                            |
| `app_temp`   | Nhiệt độ cảm nhận (độ C), kiểu dữ liệu `float64`.                                           |
| `rh`         | Độ ẩm tương đối (%) của không khí, kiểu dữ liệu `int64`.                                     |
| `wind_spd`   | Tốc độ gió (m/s), kiểu dữ liệu `float64`.                                                   |
| `wind_dir`   | Hướng gió (độ từ 0 đến 360), kiểu dữ liệu `int64`.                                           |
| `pres`       | Áp suất khí quyển (hPa), kiểu dữ liệu `int64`.                                              |
| `vis`        | Tầm nhìn (m), kiểu dữ liệu `float64`.                                                      |
| `clouds`     | Tỷ lệ mây che phủ (%), kiểu dữ liệu `int64`.                                                |
| `precip`     | Lượng mưa (mm), kiểu dữ liệu `float64`.                                                     |
| `uv`         | Chỉ số UV (độ mạnh của tia cực tím), kiểu dữ liệu `float64`.                                |
| `dewpt`      | Nhiệt độ sương (độ C), kiểu dữ liệu `float64`.                                              |

In [18]:
df_air = pd.read_csv('../data_file/hanoi_air_quality_history.csv')
df_air.describe()

,aqi,pm25,pm10,o3,so2,no2,co
count,24058.000000,24058.000000,24058.000000,24058.000000,24058.000000,24058.000000,24058.000000
mean,122.510142,49.623707,70.304086,53.579088,63.053005,31.646068,604.148516
std,58.422409,40.212509,76.289851,44.069603,62.617734,34.612708,934.466316
min,4.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
25%,80.000000,25.255000,34.000000,22.700000,20.000000,12.000000,122.600000
50%,110.000000,39.000000,52.000000,41.800000,54.000000,20.000000,205.600000
75%,155.000000,59.000000,81.300000,72.700000,88.700000,36.900000,711.000000
max,500.000000,457.000000,1103.000000,700.000000,721.000000,625.000000,15956.200000



| Cột        | Mô tả                                                                                       |
|------------|---------------------------------------------------------------------------------------------|
| `datetime` | Thời gian ghi dữ liệu, dạng chuỗi (object), chứa thông tin về ngày và giờ.                 |
| `aqi`      | Chỉ số chất lượng không khí (Air Quality Index), kiểu dữ liệu `int64`.                      |
| `pm25`     | Nồng độ bụi mịn PM2.5 (μg/m³), kiểu dữ liệu `float64`.                                      |
| `pm10`     | Nồng độ bụi PM10 (μg/m³), kiểu dữ liệu `float64`.                                          |
| `o3`       | Nồng độ Ozone (μg/m³), kiểu dữ liệu `float64`.                                              |
| `so2`      | Nồng độ Sulfur Dioxide (μg/m³), kiểu dữ liệu `float64`.                                    |
| `no2`      | Nồng độ Nitrogen Dioxide (μg/m³), kiểu dữ liệu `float64`.                                  |
| `co`       | Nồng độ Carbon Monoxide (μg/m³), kiểu dữ liệu `float64`.                                   |


### Merge dữ liệu từ hai DataFrame

Để có thể dự đoán PM2.5 dựa trên các yếu tố thời tiết, chúng ta cần kết hợp dữ liệu từ hai DataFrame lại với nhau theo cột thời gian.

In [4]:
# Kiểm tra format của cột datetime trong cả hai DataFrame
print("Format datetime trong df_weather:")
print(df_weather['datetime'].head())
print("\nFormat datetime trong df_air:")
print(df_air['datetime'].head())

# Kiểm tra kiểu dữ liệu
print(f"\nKiểu dữ liệu datetime trong df_weather: {df_weather['datetime'].dtype}")
print(f"Kiểu dữ liệu datetime trong df_air: {df_air['datetime'].dtype}")

# Kiểm tra số lượng bản ghi
print(f"\nSố bản ghi df_weather: {len(df_weather)}")
print(f"Số bản ghi df_air: {len(df_air)}")

# Kiểm tra khoảng thời gian
print(f"\nKhoảng thời gian df_weather: từ {df_weather['datetime'].min()} đến {df_weather['datetime'].max()}")
print(f"Khoảng thời gian df_air: từ {df_air['datetime'].min()} đến {df_air['datetime'].max()}")

Format datetime trong df_weather:
0    2022-12-31:17
1    2022-12-31:18
2    2022-12-31:19
3    2022-12-31:20
4    2022-12-31:21
Name: datetime, dtype: object

Format datetime trong df_air:
0    2023-01-30:17
1    2023-01-30:16
2    2023-01-30:15
3    2023-01-30:14
4    2023-01-30:13
Name: datetime, dtype: object

Kiểu dữ liệu datetime trong df_weather: object
Kiểu dữ liệu datetime trong df_air: object

Số bản ghi df_weather: 24024
Số bản ghi df_air: 24058

Khoảng thời gian df_weather: từ 2022-12-31:17 đến 2025-10-30:16
Khoảng thời gian df_air: từ 2022-12-31:17 đến 2025-10-30:17


In [5]:
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'], format='%Y-%m-%d:%H')
df_air['datetime'] = pd.to_datetime(df_air['datetime'], format='%Y-%m-%d:%H')

# Check định dạng sau chuyển đổi
print(f"df_weather datetime type: {df_weather['datetime'].dtype}")
print(f"df_air datetime type: {df_air['datetime'].dtype}")

df_weather datetime type: datetime64[ns]
df_air datetime type: datetime64[ns]


In [6]:
merged_df = pd.merge(df_weather, df_air, on='datetime', how='inner')

In [7]:
print("\nThông tin DataFrame sau khi merge:")
print(f"Số bản ghi: {len(merged_df)}")
print(f"Số cột: {len(merged_df.columns)}")
print(f"Kích thước: {merged_df.shape}")

print(f"\nKhoảng thời gian dữ liệu: từ {merged_df['datetime'].min()} đến {merged_df['datetime'].max()}")

# Hiển thị thông tin về DataFrame đã merge
print("\nThông tin chi tiết DataFrame đã merge:")
merged_df.info()
merged_df.head()


Thông tin DataFrame sau khi merge:
Số bản ghi: 24024
Số cột: 19
Kích thước: (24024, 19)

Khoảng thời gian dữ liệu: từ 2022-12-31 17:00:00 đến 2025-10-30 16:00:00

Thông tin chi tiết DataFrame đã merge:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24024 entries, 0 to 24023
Data columns (total 19 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  24024 non-null  datetime64[ns]
 1   temp      24024 non-null  float64       
 2   app_temp  24024 non-null  float64       
 3   rh        24024 non-null  int64         
 4   wind_spd  24024 non-null  float64       
 5   wind_dir  24024 non-null  int64         
 6   pres      24024 non-null  int64         
 7   vis       24017 non-null  float64       
 8   clouds    24024 non-null  int64         
 9   precip    24024 non-null  float64       
 10  uv        24024 non-null  float64       
 11  dewpt     24024 non-null  float64       
 12  aqi       24024 non-null  int64        

,datetime,temp,app_temp,rh,wind_spd,wind_dir,pres,vis,clouds,precip,uv,dewpt,aqi,pm25,pm10,o3,so2,no2,co
0,2022-12-31 17:00:00,14.8,14.8,73,0.66,310,1024,10.0,87,0.0,0.0,10.0,155,59.0,73.8,55.7,62.3,9.0,224.5
1,2022-12-31 18:00:00,14.6,14.6,75,1.00,360,1023,10.0,87,0.0,0.0,10.2,171,71.0,88.8,56.0,59.0,6.0,206.0
2,2022-12-31 19:00:00,14.3,14.3,79,1.00,345,1023,10.0,83,0.0,0.0,10.7,179,77.0,96.3,54.0,58.0,6.0,203.7
3,2022-12-31 20:00:00,14.1,14.1,82,1.00,335,1023,10.0,79,0.0,0.0,11.0,199,92.0,115.0,52.0,57.0,6.0,201.3
4,2022-12-31 21:00:00,13.8,13.8,86,1.00,320,1022,10.0,75,0.0,0.0,11.5,161,64.0,80.0,50.0,56.0,6.0,199.0


In [20]:
merged_df.select_dtypes(include=[np.number]).describe()

,temp,app_temp,rh,wind_spd,wind_dir,pres,vis,clouds,precip,uv,dewpt,aqi,pm25,pm10,o3,so2,no2,co
count,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24017.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000,24024.000000
mean,25.323810,28.380303,78.461455,1.679912,175.933899,1009.663836,10.252321,69.429820,0.318248,1.321932,20.970051,122.523310,49.626878,70.314361,53.604508,63.055253,31.642070,603.981144
std,5.455107,8.461583,14.985092,0.847098,97.484502,7.398373,2.674902,32.881189,1.143260,1.918951,5.954365,58.418296,40.199839,76.315692,44.090166,62.621304,34.618823,934.567183
min,7.000000,6.600000,17.000000,0.000000,0.000000,983.000000,0.000000,0.000000,0.000000,0.000000,-6.900000,4.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
25%,21.500000,21.800000,69.000000,1.000000,116.000000,1004.000000,10.000000,50.000000,0.000000,0.000000,17.700000,80.000000,25.325000,34.000000,22.700000,20.000000,12.000000,122.600000
50%,26.200000,27.100000,83.000000,1.660000,142.000000,1009.000000,10.000000,79.000000,0.000000,0.600000,23.300000,110.000000,39.000000,52.000000,41.800000,54.000000,20.000000,205.400000
75%,29.200000,35.200000,91.000000,2.000000,245.000000,1015.000000,10.000000,100.000000,0.000000,2.100000,25.400000,155.000000,59.000000,81.300000,72.700000,88.700000,36.900000,710.725000
max,40.000000,52.700000,100.000000,12.400000,360.000000,1033.000000,16.000000,100.000000,40.000000,12.100000,29.600000,500.000000,457.000000,1103.000000,700.000000,721.000000,625.000000,15956.200000


In [9]:
# 1. Thống kê missing value
print('=' * 40)
missing_count = merged_df.isna().sum()
missing_pct = (merged_df.isna().mean() * 100).round(2)
missing_summary = (
    pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
    .sort_values("missing_pct", ascending=False)
)
print("Tóm tắt missing values (chỉ hiển thị cột có thiếu):")
print(missing_summary[missing_summary.missing_count > 0])
print(f"\nTổng số hàng có ít nhất 1 giá trị thiếu: {merged_df.isna().any(axis=1).sum()} / {len(merged_df)}")

# 2. Thống kê trùng lặp
print('=' * 40)
dup_rows = merged_df.duplicated().sum()
print(f"Số dòng trùng lặp tuyệt đối: {dup_rows}")

# 3. Thống kê Outliers theo IQR
print('=' * 40)
numeric_cols = merged_df.select_dtypes(include=[np.number]).columns.tolist()

def iqr_bounds(s: pd.Series):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        return None, None
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_stats = []
for col in numeric_cols:
    col_series = merged_df[col].dropna()
    low, up = iqr_bounds(col_series)
    if low is None:
        outlier_stats.append((col, 0, 0.0, np.nan, np.nan))
        continue
    mask = (merged_df[col] < low) | (merged_df[col] > up)
    cnt = int(mask.sum())
    pct = round(100 * cnt / len(merged_df), 2)
    outlier_stats.append((col, cnt, pct, low, up))

outlier_summary = pd.DataFrame(
    outlier_stats, columns=["feature", "outlier_count", "outlier_pct", "lower_bound", "upper_bound"]
).sort_values("outlier_pct", ascending=False)

print("\nTop 10 biến có outliers nhiều nhất:")
print(outlier_summary.head(10))

any_outlier_mask = pd.Series(False, index=merged_df.index)
for col in numeric_cols:
    low, up = iqr_bounds(merged_df[col].dropna())
    if low is None:
        continue
    any_outlier_mask |= (merged_df[col] < low) | (merged_df[col] > up)

print(f"\nSố hàng có ít nhất 1 outlier: {any_outlier_mask.sum()} / {len(merged_df)}")

Tóm tắt missing values (chỉ hiển thị cột có thiếu):
     missing_count  missing_pct
vis              7         0.03

Tổng số hàng có ít nhất 1 giá trị thiếu: 7 / 24024
Số dòng trùng lặp tuyệt đối: 0

Top 10 biến có outliers nhiều nhất:
     feature  outlier_count  outlier_pct  lower_bound  upper_bound
17        co           2771        11.53    -759.5875    1592.9125
16       no2           2163         9.00     -25.3500      74.2500
12      pm25           1743         7.26     -25.1875     109.5125
13      pm10           1743         7.26     -36.9500     152.2500
9         uv           1190         4.95      -3.1500       5.2500
14        o3           1070         4.45     -52.3000     147.7000
10     dewpt            565         2.35       6.1500      36.9500
3   wind_spd            525         2.19      -0.5000       3.5000
15       so2            416         1.73     -83.0500     191.7500
11       aqi            324         1.35     -32.5000     267.5000

Số hàng có ít nhất 1 outli

In [ ]:
# Xử lý biến hướng gió
# Chuẩn hoá 360 -> 0
merged_df['wind_dir'] = merged_df['wind_dir'] % 360
# Tạo encoding dạng vòng tròn
merged_df['wind_dir_rad'] = np.deg2rad(merged_df['wind_dir'])
merged_df['wind_dir_sin'] = np.sin(merged_df['wind_dir_rad'])
merged_df['wind_dir_cos'] = np.cos(merged_df['wind_dir_rad'])